In [1]:
import os
import pickle
from tqdm import tqdm
from glob import glob

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, load_metric

import torch
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments,
                          DataCollatorForSeq2Seq, Trainer, pipeline)
from peft import (LoraConfig, get_peft_model, TaskType,
                  PeftModel, PeftConfig)

import warnings
warnings.filterwarnings('ignore')

import logging
logging.basicConfig(level = logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)

In [2]:
class CFG:
    wandb = False
    report_to = None
    lab_assignment = 3
    _wandb_kernel = "temuujin"

    debug = False
    num_workers = 12

    prefix_val = "summarize: "
    output_dir = "processed_data"
    model_save_dir = "PEFT_T5"

    tokenizer_name = "google-t5/t5-base"
    model_name = "google-t5/t5-base"

    project = 'NUM-Machine-Learning-Lab-3'
    name = "Lab 3 Model Training - Text Summarization T5 base, 5 epochs"

    config = {
        "output_dir": "t5_small_lab3_finetune_PEFT",
        "group": model_name,
        "learning_rate": 2e-5,
        "weight_decay": 1e-3,
        'num_train_epochs': 5,
        "train_batch_size": 2,
        "eval_batch_size": 8,
        "max_seq_length": 1024,
        "overwrite_output_dir": True,
        "reprocess_input_data": True,
        "fp16": True
    }

    test_size = 0.2

    train = True
    eval = True

    eval_metric = "rouge"

    early_stopping_patience = 15

if CFG.debug:
    CFG.config['num_train_epochs'] = 2

if CFG.wandb:
    os.environ["WANDB_SILENT"] = "True"
    CFG.report_to = "wandb"

    import wandb
    wandb.login()

    run = wandb.init(
        project = CFG.project,
        name = CFG.name,
        config = CFG.config
    )

config = CFG.config

In [3]:
df_names = glob("../Lab 2/processed_dfs/*.parquet")
search_terms = '\n'.join([os.path.basename(filename).split('_')[1] for filename in df_names])

df = pd.DataFrame()
for df_name in tqdm(df_names):
    df = pd.concat([df, pd.read_parquet(df_name)[['title', 'abstract']]])

df.drop_duplicates(inplace = True)
df.reset_index(drop = True, inplace = True)

df.rename(columns = {'title': 'target_text', 'abstract': 'input_text'}, inplace = True)
df['prefix'] = CFG.prefix_val

os.makedirs(CFG.output_dir, exist_ok = True)
output_filename = os.path.join(CFG.output_dir, "arxiv_title_generation.parquet")
df.to_parquet(output_filename)

100%|██████████| 12/12 [00:00<00:00, 16.46it/s]


In [4]:
test_size = CFG.test_size
test_df = df.sample(frac = test_size, random_state = 1970)
train_df = df.drop(index = test_df.index)

print(f"Training instance count: {len(train_df)}\nTest instance count: {len(test_df)}\n")

train_dataset = Dataset.from_dict(train_df)
test_dataset = Dataset.from_dict(test_df)
arxiv_title_dict = DatasetDict({"train": train_dataset,"test": test_dataset})

output_filename = os.path.join(CFG.output_dir, "arxiv_title_generation_dataset")
arxiv_title_dict.save_to_disk(output_filename)

print(arxiv_title_dict)

Training instance count: 11570
Test instance count: 2892



Saving the dataset (0/1 shards):   0%|          | 0/11570 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2892 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['target_text', 'input_text', 'prefix'],
        num_rows: 11570
    })
    test: Dataset({
        features: ['target_text', 'input_text', 'prefix'],
        num_rows: 2892
    })
})


In [5]:
N = 7
random_indices = np.random.randint(0, 2892, size = N)
actual_titles = np.asarray(arxiv_title_dict['test']['target_text'])[random_indices]
abstracts = np.asarray(arxiv_title_dict['test']['input_text'])[random_indices]

In [6]:
model_save_paths = {
    "T5-Base 5 epochs (PEFT)": "PEFT_T5/test_v10",
    "T5-Small 15 epochs (PEFT)": "PEFT_T5/test_v9",
    "T5-Efficient-Mini 60 epochs (PEFT)": "PEFT_T5/test_v7",
    "T5-Efficient-Mini 90 epochs (PEFT)": "PEFT_T5/test_v8",
}

In [7]:
device = 'cuda'

for idx, (model_name, model_save_path) in enumerate(model_save_paths.items()):
    peft_model_id = model_save_path

    torch.cuda.empty_cache()

    config = PeftConfig.from_pretrained(peft_model_id)
    tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

    model = AutoModelForSeq2SeqLM.from_pretrained(config.base_model_name_or_path)
    model = PeftModel.from_pretrained(model, peft_model_id)

    model = model.to(device)
    model.eval()

    with torch.no_grad():
        inputs = [CFG.prefix_val + doc for doc in abstracts]
        dencoded_results = []

        for input_text in inputs:
            model_input = tokenizer(input_text, return_tensors = 'pt')
            input_ids = torch.tensor(model_input['input_ids']).to(device)
            encoded_result = model.generate(input_ids = input_ids, max_new_tokens = 10)
            decoded_result = tokenizer.batch_decode(encoded_result.detach().cpu().numpy(), skip_special_tokens = True)[0]

            dencoded_results.append(decoded_result)

    torch.cuda.empty_cache()

    with open("output.txt", "a") as file:
        file.write(f"**{model_name}**:\n")
        for i in range(N):
            file.write(f"Test index: {random_indices[i]}\n\n")
            file.write(f"Actual Title: {actual_titles[i]}\n\n")
            file.write(f"Predicted Title: {dencoded_results[i]}\n\n")
            file.write("*"*30)
            file.write("\n")

In [8]:
torch.cuda.empty_cache()

full_finetuned_model = pipeline("summarization", model = "./t5_small_lab3_finetune/checkpoint-11500")

with torch.no_grad():
    batch_input = [CFG.prefix_val + input_text for input_text in abstracts]
    pred = full_finetuned_model(batch_input)
    pred = [pred_text['summary_text'] for pred_text in pred]

    with open("output.txt", "a") as file:
        file.write("**T5-Small 1.5 epochs (Full)**:\n")
        for i in range(N):
            file.write(f"Test index: {random_indices[i]}\n\n")
            file.write(f"Actual Title: {actual_titles[i]}\n\n")
            file.write(f"Predicted Title: {pred[i]}\n\n")
            file.write("*"*30)
            file.write("\n")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers
